# 1. Carga y descompresión BREAST_ROI_TRAIN


In [27]:
from google.colab import files
import zipfile
import os

uploaded = files.upload()

zip_path = "clasification-roi.zip"
extract_path = "dataset"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("Dataset descomprimido correctamente")

Saving clasification-roi.zip to clasification-roi (1).zip
Dataset descomprimido correctamente


# 2. Verificación de estructura


In [34]:
import os

DATASET_ROOT = "/content/dataset"

print("Contenido raíz:")
print(os.listdir(DATASET_ROOT))

print("\nEjemplo pacientes Benign:")
print(os.listdir(f"{DATASET_ROOT}/train/Benign")[:5])

Contenido raíz:
['train', 'test', 'val']

Ejemplo pacientes Benign:
['BreaDM-Be-1801', 'BreaDM-Be-1811', 'BreaDM-Be-1911', 'BreaDM-Be-1816', 'BreaDM-Be-1818']


#3. BLOQUE 3 — Constantes y funciones reutilizables


In [35]:
import os
import csv
import pandas as pd

DATASET_ROOT = "/content/dataset"

PRE   = "VIBRANT"
EARLY = "VIBRANT+C2"
LATE  = "VIBRANT+C7"

CLASSES = ["Benign", "Malignant"]


def generate_csv(split, output_name):
    """
    Una fila por slice.
    Columnas: patient, label, pre_contrast, post_early, post_late.
    Rutas relativas (sin /content/dataset/).
    Sin columna split.
    """
    rows = []

    for label in CLASSES:

        class_path = os.path.join(DATASET_ROOT, split, label)

        if not os.path.exists(class_path):
            print(f"  [ADVERTENCIA] No encontrado: {class_path}")
            continue

        for patient in os.listdir(class_path):

            patient_path = os.path.join(class_path, patient)

            pre_path   = os.path.join(patient_path, PRE)
            early_path = os.path.join(patient_path, EARLY)
            late_path  = os.path.join(patient_path, LATE)

            if not os.path.exists(pre_path):   continue
            if not os.path.exists(early_path): continue
            if not os.path.exists(late_path):  continue

            pre_images   = sorted([f for f in os.listdir(pre_path)   if f.endswith(".jpg")])
            early_images = sorted([f for f in os.listdir(early_path) if f.endswith(".jpg")])
            late_images  = sorted([f for f in os.listdir(late_path)  if f.endswith(".jpg")])

            for pre_img, early_img, late_img in zip(pre_images, early_images, late_images):

                rows.append({
                    "patient":      patient,
                    "label":        label,
                    "pre_contrast": f"{split}/{label}/{patient}/{PRE}/{pre_img}",
                    "post_early":   f"{split}/{label}/{patient}/{EARLY}/{early_img}",
                    "post_late":    f"{split}/{label}/{patient}/{LATE}/{late_img}"
                })

    with open(output_name, "w", newline="") as f:
        writer = csv.DictWriter(
            f,
            fieldnames=["patient", "label", "pre_contrast", "post_early", "post_late"]
        )
        writer.writeheader()
        writer.writerows(rows)

    print(f"\n✅ {output_name} — {len(rows)} slices generadas")
    return pd.read_csv(output_name)


def validate_basic_info(df, name):
    print(f"\n📊 {name}")
    print("  Slices totales:   ", len(df))
    print("  Pacientes únicos: ", df["patient"].nunique())
    print("  Distribución de clases:")
    print(df["label"].value_counts().to_string())

#BLOQUE 4 — Generación de los 3 CSVs y validación


In [36]:
splits = {
    "train": "BREAST_ROI_TRAIN.csv",
    "test":  "BREAST_ROI_TEST.csv",
    "val":   "BREAST_ROI_VAL.csv"
}

dataframes = {}

for split, output_name in splits.items():
    print(f"\n🔍 Escaneando split: {split}...")
    df = generate_csv(split, output_name)
    validate_basic_info(df, output_name)
    dataframes[split] = df


🔍 Escaneando split: train...

✅ BREAST_ROI_TRAIN.csv — 1202 slices generadas

📊 BREAST_ROI_TRAIN.csv
  Slices totales:    1202
  Pacientes únicos:  166
  Distribución de clases:
label
Malignant    875
Benign       327

🔍 Escaneando split: test...

✅ BREAST_ROI_TEST.csv — 403 slices generadas

📊 BREAST_ROI_TEST.csv
  Slices totales:    403
  Pacientes únicos:  47
  Distribución de clases:
label
Malignant    289
Benign       114

🔍 Escaneando split: val...

✅ BREAST_ROI_VAL.csv — 117 slices generadas

📊 BREAST_ROI_VAL.csv
  Slices totales:    117
  Pacientes únicos:  19
  Distribución de clases:
label
Malignant    93
Benign       24


#BLOQUE 5 — Descarga de los 3 CSVs


In [37]:
# Se descargan uno por uno con confirmación para evitar el bug de Colab
from google.colab import files
import time

for output_name in splits.values():
    print(f"⬇️  Descargando: {output_name}")
    files.download(output_name)
    time.sleep(2)  # Pausa entre descargas para que Colab no las bloquee

⬇️  Descargando: BREAST_ROI_TRAIN.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

⬇️  Descargando: BREAST_ROI_TEST.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

⬇️  Descargando: BREAST_ROI_VAL.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [38]:
import os

DATASET_ROOT = "/content/dataset"

# Tomar un paciente de ejemplo y listar sus carpetas
ejemplo = "/content/dataset/train/Benign"
primer_paciente = os.listdir(ejemplo)[0]
ruta_paciente = os.path.join(ejemplo, primer_paciente)

print(f"Paciente: {primer_paciente}")
print("Carpetas dentro:")
print(sorted(os.listdir(ruta_paciente)))

Paciente: BreaDM-Be-1801
Carpetas dentro:
['SUB1', 'SUB2', 'SUB3', 'SUB4', 'SUB5', 'SUB6', 'SUB7', 'SUB8', 'VIBRANT', 'VIBRANT+C1', 'VIBRANT+C2', 'VIBRANT+C3', 'VIBRANT+C4', 'VIBRANT+C5', 'VIBRANT+C6', 'VIBRANT+C7', 'VIBRANT+C8']


#BLOQUE — Verificación de secuencias completas


In [41]:
import os

DATASET_ROOT = "/content/dataset"

SEQUENCES = ["VIBRANT", "VIBRANT+C1", "VIBRANT+C2",
             "VIBRANT+C3", "VIBRANT+C4", "VIBRANT+C5",
             "VIBRANT+C6", "VIBRANT+C7", "VIBRANT+C8"]

CLASSES  = ["Benign", "Malignant"]
splits   = ["train", "test", "val"]

total    = 0
completos = 0
incompletos = []

for split in splits:
    for label in CLASSES:

        class_path = os.path.join(DATASET_ROOT, split, label)

        if not os.path.exists(class_path):
            continue

        for patient in os.listdir(class_path):

            total += 1
            patient_path = os.path.join(class_path, patient)

            faltantes = [
                seq for seq in SEQUENCES
                if not os.path.exists(os.path.join(patient_path, seq))
            ]

            if faltantes:
                incompletos.append({
                    "split":    split,
                    "label":    label,
                    "patient":  patient,
                    "faltantes": faltantes
                })
            else:
                completos += 1

# Resumen
print(f"✅ Pacientes completos:    {completos} / {total}")
print(f"❌ Pacientes incompletos:  {len(incompletos)} / {total}")

if incompletos:
    print("\nDetalle de pacientes con secuencias faltantes:")
    print(f"{'Split':<8} {'Label':<12} {'Paciente':<20} {'Faltantes'}")
    print("-" * 70)
    for p in incompletos:
        print(f"{p['split']:<8} {p['label']:<12} {p['patient']:<20} {p['faltantes']}")
else:
    print("\n🎉 Todos los pacientes tienen las 9 secuencias completas.")

✅ Pacientes completos:    232 / 232
❌ Pacientes incompletos:  0 / 232

🎉 Todos los pacientes tienen las 9 secuencias completas.


#Github

In [42]:
!git init

hint: Using 'master' as the name for the initial branch. This default branch name
hint: is subject to change. To configure the initial branch name to use in all
hint: of your new repositories, which will suppress this warning, call:
hint: 
hint: 	git config --global init.defaultBranch <name>
hint: 
hint: Names commonly chosen instead of 'master' are 'main', 'trunk' and
hint: 'development'. The just-created branch can be renamed via this command:
hint: 
hint: 	git branch -m <name>
Initialized empty Git repository in /content/.git/


# CONFIGURACIÓN DE GIT Y CLONACIÓN

In [43]:

import os
import getpass

# Datos de GitHub
USER_NAME  = "PandoraRiot"
USER_EMAIL = "e.garcia1565@pascualbravo.edu.co"
REPO_NAME  = "breast-cancer-dce-mri-classification"

# Configuración global de Git
!git config --global user.email "{USER_EMAIL}"
!git config --global user.name "{USER_NAME}"

# Autenticación segura con Token
token    = getpass.getpass("🔑 Pega aquí tu Personal Access Token (PAT): ")
repo_url = f"https://{token}@github.com/{USER_NAME}/{REPO_NAME}.git"

# Clonar el repositorio si no existe
if not os.path.exists(f'/content/{REPO_NAME}'):
    print(f"📥 Clonando {REPO_NAME}...")
    !git clone {repo_url}
else:
    print(f"✅ El repositorio '{REPO_NAME}' ya existe en /content.")

# Entrar al repositorio
%cd /content/{REPO_NAME}

# Instalar librerías
if os.path.exists('requirements.txt'):
    !pip install -r requirements.txt
else:
    print("⚠️ No se encontró requirements.txt, instalando librerías base...")
    !pip install torch torchvision numpy matplotlib scikit-learn seaborn pandas

print(f"\n🚀 Entorno configurado. Trabajando en: {os.getcwd()}")

🔑 Pega aquí tu Personal Access Token (PAT): ··········
📥 Clonando breast-cancer-dce-mri-classification...
Cloning into 'breast-cancer-dce-mri-classification'...
/content/breast-cancer-dce-mri-classification
⚠️ No se encontró requirements.txt, instalando librerías base...

🚀 Entorno configurado. Trabajando en: /content/breast-cancer-dce-mri-classification


#SUBIR NOTEBOOK A GITHUB

In [47]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [48]:
import os

result = !find /content/drive -name "dataset_preparation.ipynb" 2>/dev/null
print(result)

['/content/drive/MyDrive/Colab Notebooks/dataset_preparation.ipynb']


In [45]:
import shutil

NOTEBOOK_NAME = "dataset_preparation.ipynb"
DRIVE_PATH    = f"/content/drive/MyDrive/Colab Notebooks/{NOTEBOOK_NAME}"

# Copiar desde Drive al repositorio
shutil.copy(DRIVE_PATH, f"/content/{REPO_NAME}/{NOTEBOOK_NAME}")

# Commit y push
!git add {NOTEBOOK_NAME}
!git commit -m "feat: notebook de preparación y generación de dataset DCE-MRI, se realiza la creación de 3 CSVs, divididos en train, test, val, cada dataset es extraído del "
!git push {repo_url}

print("✅ Notebook subido correctamente a GitHub.")

FileNotFoundError: [Errno 2] No such file or directory: '/content/dataset_preparation.ipynb'